In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]          # dev/notebooks -> repository root
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "dev")]


# SCIO offline evidence pipeline
This notebook inventories raw observations and measures competing transform hypotheses. It deliberately does not label opaque bytes as a spectrum.

In [ ]:
from pathlib import Path
from scio import corpus
from scio_offline import evidence
from scio.reference import inspect_reference_csv

In [ ]:
roots = [ROOT / '01_rawdata/log_extracted',
         ROOT / '01_rawdata/scan_json',
         ROOT / '01_rawdata/scan_json_calibration']
records, errors = corpus.build_corpus(roots)
len(records), errors[:3]

In [ ]:
report = evidence.corpus_report(records)
report['summary'], report['interpretation']

In [ ]:
out = ROOT / 'dev' / 'analysis_output'
corpus.write_index(records, out / 'corpus_index.json', errors)
evidence.write_report(report, out / 'transform_evidence_report.json')

## Look at the raw bytes

Render each blob body as a 16-byte-wide raster. High entropy, no visible
structure - this is the encryption the key would undo. Point `SCAN` at any
canonical record in `01_rawdata/scans/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scio import session
from scio_offline.decode import entropy, split_blob

SCAN = sorted((ROOT / '01_rawdata' / 'scans').glob('*.json'))[0]
rec = session.load_record(SCAN)
blobs, _ = session.record_blobs(rec)

fig, axes = plt.subplots(1, len(blobs), figsize=(4 * len(blobs), 3), squeeze=False)
for ax, (name, blob) in zip(axes[0], blobs.items()):
    b = split_blob(blob)
    ax.imshow(np.frombuffer(b.body, np.uint8).reshape(-1, 16), aspect="auto", cmap="viridis")
    ax.set_title(f"{name}  entropy {entropy(b.body):.2f} bit/B")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"{rec['annotation']['name']}  ({SCAN.name})")
plt.tight_layout(); plt.show()

## Next gate
Collect the controlled replicate matrix in `documentation/EVIDENCE.md`. Only create a validated decoder profile after cross-session and held-out spectral tests pass.